In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#  Normal Decision tree on dataset without any pipeline

In [ ]:
df = pd.read_csv('train.csv');
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True);
df


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S
887,1,1,female,19.0,0,0,30.0000,S
888,0,3,female,NaN,1,2,23.4500,S
889,1,1,male,26.0,0,0,30.0000,C


In [ ]:
df.isnull().sum()

,0
Survived,0
Pclass,0
Sex,0
Age,177
SibSp,0
Parch,0
Fare,0
Embarked,2


In [ ]:
# Age and Embarked : Simple imputer,    Sex and Embarked: One hot encoding

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,Y_train,Y_test = train_test_split(df.drop(columns=['Survived']),df['Survived'],test_size=0.3,random_state=42);


In [ ]:
X_train

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
445,1,male,4.0,0,2,81.8583,S
650,3,male,NaN,0,0,7.8958,S
172,3,female,1.0,1,1,11.1333,S
450,2,male,36.0,1,2,27.7500,S
314,2,male,43.0,1,1,26.2500,S
...,...,...,...,...,...,...,...
106,3,female,21.0,0,0,7.6500,S
270,1,male,NaN,0,0,31.0000,S
860,3,male,41.0,2,0,14.1083,S
435,1,female,14.0,1,2,120.0000,S


In [ ]:
from sklearn.impute import SimpleImputer;               # simple imputer

si_age = SimpleImputer();
si_age.fit(X_train[['Age']]);

X_train_age_transfomed = si_age.transform(X_train[['Age']]);
X_test_age_transfomed = si_age.transform(X_test[['Age']]);

si_embarked = SimpleImputer(strategy='most_frequent');
si_embarked.fit(X_train[['Embarked']]);

X_train_embarked_transformed = si_embarked.transform(X_train[['Embarked']]);
X_test_embarked_transformed = si_embarked.transform(X_test[['Embarked']]);



In [ ]:
# one hot encoding
from sklearn.preprocessing import OneHotEncoder
ohe_sex = OneHotEncoder(sparse_output=False,drop='first',handle_unknown='ignore');
ohe_embarked = OneHotEncoder(sparse_output=False,drop='first',handle_unknown='ignore');

ohe_sex.fit(X_train[['Sex']]);
ohe_embarked.fit(X_train_embarked_transformed);

X_train_sex_transfomred = ohe_sex.transform(X_train[['Sex']]);
X_test_sex_transformed = ohe_sex.transform(X_test[['Sex']]);

X_train_embarked_transformed = ohe_embarked.transform(X_train_embarked_transformed);
X_test_embarked_transformed = ohe_embarked.transform(X_test_embarked_transformed);




In [ ]:
X_trained_remaining = X_train.drop(columns=['Sex','Age','Embarked']);
X_test_remaining = X_test.drop(columns=['Sex','Age','Embarked']);



,Pclass,SibSp,Parch,Fare
709,3,1,1,15.2458
439,2,0,0,10.5000
840,3,0,0,7.9250
720,2,0,1,33.0000
39,3,1,0,11.2417
...,...,...,...,...
821,3,0,0,8.6625
633,1,0,0,0.0000
456,1,0,0,26.5500
500,3,0,0,8.6625


In [ ]:
X_train_transfomed = np.concatenate((X_train_sex_transfomred,X_train_embarked_transformed,X_train_age_transfomed,X_trained_remaining),axis=1);
X_test_transfomed = np.concatenate((X_test_sex_transformed,X_test_embarked_transformed,X_test_age_transfomed,X_test_remaining),axis=1);

X_train_transfomed.shape


(623, 8)

In [ ]:
from sklearn.tree import DecisionTreeClassifier;

clf = DecisionTreeClassifier();

clf.fit(X_train_transfomed,Y_train);

Y_predicted = clf.predict(X_test_transfomed);


In [ ]:
from sklearn.metrics import accuracy_score;

print(accuracy_score(Y_predicted,Y_test));

0.7611940298507462
